In [48]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.optimize import curve_fit

from uvx import UVXReader

In [49]:
POL_MAP = {
    "RR": -1,
    "LL": -2,
    "RL": -3,
    "LR": -4,
}

In [50]:
def brightness_temperature(
    S0_jy,
    theta_mas,
    freq_hz,
):
    """
    Observed-frame brightness temperature.
    """

    freq_ghz = freq_hz / 1e9

    Tb = (
        1.22e12
        * S0_jy
        / (freq_ghz**2 * theta_mas**2)
    )

    return Tb

In [51]:
def read_uvx(
    uvx_file,
    pol="RR",
    if_index=0,
    channel_index=0,
):

    with UVXReader(uvx_file) as uvx:

        data = uvx[0:len(uvx)]

        target_pol = POL_MAP[pol]

        stokes_index = None

        for i, s in enumerate(uvx.stokes):

            sval = s.value if hasattr(s, "value") else int(s)

            if sval == target_pol:
                stokes_index = i
                break

        if stokes_index is None:
            raise RuntimeError(
                f"{pol} not found"
            )

        u = data["header"]["u_wave"]
        v = data["header"]["v_wave"]

        baseline = (
            np.sqrt(u*u + v*v)
            * uvx.hdr.freq0
        )

        re = data["complex"]["re"][
            :, if_index, channel_index, stokes_index
        ]

        im = data["complex"]["im"][
            :, if_index, channel_index, stokes_index
        ]

        wt = data["complex"]["wt"][
            :, if_index, channel_index, stokes_index
        ]

        amp = np.sqrt(re**2 + im**2)

        good = (
            np.isfinite(baseline)
            & np.isfinite(amp)
            & np.isfinite(wt)
            & (wt > 0)
        )

        baseline = baseline[good]
        amp = amp[good]

        tl1 = data["header"]["tlsc1"][good]
        tl2 = data["header"]["tlsc2"][good]

        freq = uvx.hdr.freq0

    return (
        baseline,
        amp,
        tl1,
        tl2,
        freq,
    )

In [52]:
def count_unique_baselines(
    tl1,
    tl2,
):

    pairs = set()

    for a, b in zip(tl1, tl2):

        pair = tuple(sorted((int(a), int(b))))

        pairs.add(pair)

    return len(pairs)

In [53]:
def gaussian_visibility(B, S0, theta_rad):
    """
    Circular Gaussian visibility.

    B         baseline in wavelengths
    S0        zero-spacing flux density [Jy]
    theta_rad FWHM [rad]
    """

    return S0 * np.exp(
        -(np.pi * theta_rad * B) ** 2
        / (4.0 * np.log(2.0))
    )

In [54]:
def fit_gaussian(
    baseline,
    amp,
):

    order = np.argsort(baseline)

    baseline = baseline[order]
    amp = amp[order]

    S0_guess = np.max(amp)

    theta_guess_mas = 0.05

    theta_guess_rad = (
        theta_guess_mas
        / 206265000.0
    )

    popt, pcov = curve_fit(
        gaussian_visibility,
        baseline,
        amp,
        p0=[
            S0_guess,
            theta_guess_rad,
        ],
        bounds=(
            [0.0, 0.0],
            [np.inf, np.inf]
        ),
        maxfev=10000,
    )

    S0_fit = popt[0]
    theta_rad = popt[1]

    sigma_S0 = np.sqrt(
        np.abs(pcov[0,0])
    )

    sigma_theta_rad = np.sqrt(
        np.abs(pcov[1,1])
    )

    theta_mas = (
        theta_rad
        * 206265000.0
    )

    sigma_theta_mas = (
        sigma_theta_rad
        * 206265000.0
    )

    return (
        S0_fit,
        sigma_S0,
        theta_rad,
        sigma_theta_rad,
        theta_mas,
        sigma_theta_mas,
        pcov,
    )

In [55]:
def save_fit_plot(
    outfile,
    baseline,
    amp,
    S0,
    theta_rad,
):

    xfit = np.linspace(
        baseline.min(),
        baseline.max(),
        500,
    )

    yfit = gaussian_visibility(
        xfit,
        S0,
        theta_rad,
    )

    plt.figure(figsize=(8,6))

    plt.scatter(
        baseline,
        amp,
        s=10,
        label="data",
    )

    plt.plot(
        xfit,
        yfit,
        lw=2,
        label="Gaussian fit",
    )

    plt.xlabel("Baseline (lambda)")
    plt.ylabel("Correlated flux density (Jy)")
    plt.legend()

    plt.tight_layout()

    plt.savefig(
        outfile,
        dpi=200,
    )

    plt.close()

In [56]:
def brightness_temperature_error(
    Tb,
    S0,
    S0_err,
    theta,
    theta_err,
):

    frac = np.sqrt(
        (S0_err / S0)**2
        +
        (
            2.0 * theta_err / theta
        )**2
    )

    return Tb * frac

In [57]:
def parse_uvx_filename(path):

    name = os.path.basename(path)

    stem = name.replace(".uvx", "")

    parts = stem.split("_")

    mode = parts[0]
    session = parts[1]
    band = parts[2]
    date = parts[3]

    return {
        "mode": mode,
        "session": session,
        "band": band,
        "date": date,
    }

In [58]:
def choose_pol(band):

    if band in ["C", "K"]:
        return "LL"

    if band == "L":
        return "RR"
    
    if band == "P":
        return "RR"

    raise ValueError(
        f"Unknown band {band}"
    )

In [59]:
ROOT_DIR = r"i:\AGN\OJ287\CALIB"

OUT_DIR = "gaussian_fits"

os.makedirs(
    OUT_DIR,
    exist_ok=True,
)

rows = []

uvx_files = glob.glob(
    os.path.join(
        ROOT_DIR,
        "**",
        "*.uvx",
    ),
    recursive=True,
)

print(
    f"Found {len(uvx_files)} UVX files"
)

for i, uvx_file in enumerate(uvx_files, start=1):

    name = os.path.basename(uvx_file)

    print(
        f"[{i}/{len(uvx_files)}] {name}"
    )

    try:

        # ----------------------------------
        # parse filename
        # ----------------------------------

        meta = parse_uvx_filename(
            uvx_file
        )

        pol = choose_pol(
            meta["band"]
        )

        if meta["mode"] == "GVLBI":
            mode_comment = "ground_only"
        else:
            mode_comment = "space_vlbi"

        # ----------------------------------
        # read uvx
        # ----------------------------------

        (
            baseline,
            amp,
            tl1,
            tl2,
            freq,
        ) = read_uvx(
            uvx_file,
            pol=pol,
        )

        # ----------------------------------
        # count baselines
        # ----------------------------------

        nbase = count_unique_baselines(
            tl1,
            tl2,
        )

        if nbase <= 1:

            rows.append([
                name,
                meta["mode"],
                meta["session"],
                meta["band"],
                meta["date"],
                nbase,
                "SKIPPED",
                np.nan,
                np.nan,
                np.nan,
                "single_baseline",
            ])

            print(
                f"   SKIPPED ({nbase} baseline)"
            )

            continue

        # ----------------------------------
        # gaussian fit
        # ----------------------------------

        (
            S0,
            S0_err,
            theta_rad,
            theta_rad_err,
            theta_mas,
            theta_mas_err,
            pcov,
        ) = fit_gaussian(
            baseline,
            amp,
        )

        Tb = brightness_temperature(
            S0,
            theta_mas,
            freq,
        )

        Tb_err = brightness_temperature_error(
            Tb,
            S0,
            S0_err,
            theta_mas,
            theta_mas_err,
        )

        # ----------------------------------
        # save figure
        # ----------------------------------

        png_name = (
            os.path.splitext(name)[0]
            + ".png"
        )

        png_file = os.path.join(
            OUT_DIR,
            png_name,
        )

        save_fit_plot(
            png_file,
            baseline,
            amp,
            S0,
            theta_rad,
        )

        rows.append([
            name,
            meta["mode"],
            meta["session"],
            meta["band"],
            meta["date"],
            nbase,
            "OK",
            theta_mas,
            theta_mas_err,
            Tb,
            Tb_err,
            S0,
            S0_err,
            mode_comment,
        ])

        print(
            f"   OK   "
            f"Nb={nbase}   "
            f"theta={theta_mas:.4f} mas   "
            f"Tb={Tb:.2e} K"
        )

    except Exception as e:

        rows.append([
            name,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            np.nan,
            "FAILED",
            np.nan,
            np.nan,
            np.nan,
            str(e),
        ])

        print(
            f"   FAILED: {e}"
        )

# --------------------------------------
# save log
# --------------------------------------

log = pd.DataFrame(
    rows,
    columns=[
        "file",
        "mode",
        "session",
        "band",
        "date",
        "n_baselines",
        "status",
        "theta_mas",
        "theta_mas_err",
        "Tb_K",
        "Tb_K_err",
        "S0_Jy",
        "S0_Jy_err",
        "comment",
    ]
)

csv_file = os.path.join(
    OUT_DIR,
    "gaussian_fit_log.csv",
)

log.to_csv(
    csv_file,
    index=False,
)

print()
print("=" * 60)
print("Finished")
print(f"Results: {csv_file}")
print("=" * 60)

Found 136 UVX files
[1/136] RADIOASTRON_RAES03AA_C_20120427T213000_ASC_V4_cl_cl_ff_prd_tav_fav.uvx
   OK   Nb=5   theta=0.7528 mas   Tb=3.16e+11 K
[2/136] RADIOASTRON_RAES03AA_L_20120427T213000_ASC_V3_00_cl_ff_prd_tav_fav.uvx
   SKIPPED (1 baseline)
[3/136] RADIOASTRON_RAES03AB_C_20120428T213000_ASC_V4_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=1.6632 mas   Tb=5.21e+10 K
[4/136] RADIOASTRON_RAES03F_C_20120221T013000_ASC_V2_00_cl_ff_prd_tav_fav.uvx
   FAILED: LL not found
[5/136] RADIOASTRON_RAES03FW_C_20121119T081000_ASC_V2_00_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=0.0500 mas   Tb=4.30e+13 K
[6/136] RADIOASTRON_RAES03FW_L_20121119T081000_ASC_V2_cl_ff_prd_tav_fav.uvx
   OK   Nb=3   theta=3.8108 mas   Tb=1.24e+11 K
[7/136] GVLBI_RAES03I_L_20120317T150000_ASC_V3_cl_ff_prd_tav_fav.uvx
   SKIPPED (1 baseline)
[8/136] GVLBI_RAES03IK_C_20121226T220000_ASC_V2_cl_ff_prd_tav_fav.uvx
   SKIPPED (1 baseline)
[9/136] GVLBI_RAES03IK_K_20121226T220000_ASC_V2_cl_ff_prd_tav_fav.uvx
   OK   Nb=10   